![DB Academy](./Includes/images/db-academy.png)

# Lakebase Monitoring

Now that we have performed several operations in our Project across multiple Branches, let's walkthrough how to monitor each branch.

This notebook walks through the Lakebase Monitoring panel and how to use each section to understand and tune your OLTP workloads. 

## 1. Metrics Dashboards

This provides graphs for monitoring system and database metrics

How to access the Metrics Dashboards:

- Open the Lakebase App and select your project. 
- In the sidebar, navigate to the **Metrics** dashboard.   
- Choose the **branch** and **compute** from the drop-down menus.  
- Adjust the time window.
- Use **Refresh** to pull the latest metrics. 

**Now lets look at each graph to understand how to use it!**

#### 1.1 Inactive Computes and “Empty” Graphs

When graphs show only zeros or diagonal patterns, the compute may be inactive due to **scale to zero**. 

- Inactive compute cannot report metrics, so values drop to 0. 
- Diagonal line patterns mark periods where the compute is inactive. 
- If no data appears, try a different time range or generate more workload, then refresh later. 

#### 1.2 RAM Metrics

![ram.png](Includes/images/branching/lab5_ram.png)

The **RAM** graph shows allocated RAM and usage over time for the selected compute.

Explain these lines:
- **Allocated**: RAM assigned based on compute size or autoscaling. With autoscaling, this rises and falls with load; it drops to 0 when scale-to-zero idles the compute. 
- **Used**: Actual RAM consumption over time. If this line regularly hits the allocated ceiling, consider a larger compute size.   
- **Cached**: Data cached in memory by previous queries and operations.

#### 1.3 CPU Metrics

![cpu.png](Includes/images/branching/lab5_cpu.png)

The **CPU** graph shows allocated CPU and usage over time.

Discuss:
- **Allocated**: CPU assigned by compute size or autoscaling; drops to 0 when the compute scales to zero. 
- **Used**: CPU consumed, in Compute Units (CU). If this line frequently reaches the allocated limit, scale up the compute size.

#### 1.4 Connections Count

![connection_count.png](Includes/images/branching/lab5_connection_count.png)

The **Connections count** graph tracks connection usage for the selected compute.

Explain each metric:
- **Active**: Number of active connections currently executing work. High sustained active connections can indicate heavy load and slower queries. 
- **Idle**: Open but unused connections. A few are fine; many can waste resources and block new active connections. Encourage closing unnecessary idle connections.
- **Total**: Active + idle connections. This shows overall connection footprint.  
- **Max**: Maximum simultaneous connections allowed for this compute size (Postgres `max_connections`).  

How to interpret:
- When **Total** approaches **Max**, consider:
  - Increasing compute size to raise the connection limit.
  - Improving connection management (pooling, closing unused connections, avoiding long‑lived idle sessions)

#### 1.5 Database Size

![database_size.png](Includes/images/branching/lab5_databasize.png)

The **Database size** graph shows the logical size of data (tables and indexes) for the selected database or all databases on the branch.

Important notes:
- Logical size comes from Postgres-reported data size.
- Metrics only appear when compute is active; if the compute is idle, the graph shows zero even if data exists.

#### 1.6 Deadlocks

![deadlock.png](Includes/images/branching/lab5_deadlock.png)

The **Deadlocks** graph shows how many deadlocks occur over time.

Key teaching points:
- Deadlocks happen when transactions block each other in a cycle, preventing progress.
- Symptoms include errors and latency spikes.
- Use this graph to spot periods with deadlock bursts and then investigate queries and locking patterns.

#### 1.7 Row Activity (Rows)

![row.png](Includes/images/branching/lab5_rows.png)

The **Rows** graph tracks how many rows are deleted, updated, and inserted over time.

Explain:
- Metrics reset to zero on compute restart.  
- Only row-level operations (INSERT, UPDATE, DELETE) are counted; table-level operations like TRUNCATE are excluded.
- Use trends to detect spikes in writes or unusual delete patterns


#### 1.8 Replication Delay (Bytes and Seconds)

These graphs apply when using a **read replica compute**.

##### Replication delay bytes
- Shows the size (in bytes) of data sent from primary but not yet applied on the replica.  
- Larger values indicate a backlog; possible issues include limited throughput or constrained resources on the replica.

##### Replication delay seconds
- Shows the time gap (in seconds) between last commit on primary and its application on the replica.   
- High values indicate the replica is lagging behind, possibly due to network or load issues. 


#### 1.9 Local File Cache Hit Rate

![lfc.png](Includes/images/branching/lab5_lfc.png)

The **Local file cache hit rate** graph shows what percentage of read requests are served from the local file cache. 

Teaching points:
- Reads that miss both Postgres shared buffers and the local cache fall back to storage, which is slower and more expensive.   
- For OLTP workloads, target a cache hit rate of around 99% or higher.  
- If below 99%, your working set may not fit in memory; consider increasing compute size to enlarge the local file cache. 
- Local file cache can use up to 75% of compute RAM (for example, 8 GB RAM → 6 GB local file cache). 

#### 1.10 Working Set Size

![working_set.png](Includes/images/branching/lab5_workingset.png)

The **Working set size** graph helps you see how much unique data is accessed over different time windows.

Metrics shown:
- **5m**: Data accessed in the last 5 minutes.
- **15m**: Data accessed in the last 15 minutes. 
- **1h**: Data accessed in the last hour. 
- **Local file cache size**: Cache capacity determined by compute size.

How to interpret:
- For optimal performance, the working set for a given interval should fit within the local file cache.
- If working set size exceeds cache size, increase compute size to improve cache hit rate. 
- For stable workloads, compare the 1‑hour working set to the cache size to validate sizing.



## 2. Monitoring Active Queries

Now lets look at how to use the **Active queries** view to observe real-time query activity and troubleshoot performance issues in your Lakebase Postgres projects. 

How to access the Active Queries View:

- From the Lakebase App, select your project. 
- In the sidebar, select a **branch**.  
- Go to **Monitoring**.  
- Open the **Active queries** tab.  

#### 2.1 Understanding the Active Queries Table

Explain each column in the **Active queries** table and how to use it for troubleshooting. 

- **PID**  
  - Process ID of the Postgres backend executing the query.  
  - Useful for tracking or terminating a specific query if needed.  

- **Duration**  
  - How long the query has been running.  
  - You can sort by this column to quickly find long‑running queries that may be causing latency or blocking others.  

- **Query**  
  - The SQL statement currently executing.  
  - Helps you identify which application operation or user action is responsible for heavy or unexpected load.

#### 2.2 How It Works: pg_stat_activity

- The Active queries view is powered by the Postgres system view **`pg_stat_activity`**.  
- This view is available by default in Lakebase Postgres.  
- You can query `pg_stat_activity` directly using the SQL Editor or any Postgres client such as `psql`.  
- `pg_stat_activity` only shows **currently running queries**; once a query finishes, it disappears from both the system view and the Active queries UI.  

## 3. Analyzing Query Performance

Now lets look at how to use the **Query performance** view to analyze historical queries, find slow patterns, and prioritize optimization work.

How to access the Query Performance View:

- From the Lakebase App, select your project.  
- In the sidebar, select a **branch**.  
- Go to **Monitoring**.  
- Open the **Query performance** tab.  

### 3.1 Reading the Query Performance Table

![query_performance.png](Includes/images/branching/lab5_query_performance.png)

Explanation

- **Role**  
  - The Postgres role (user or application account) that executed the query.  
  - Helps you see which service or user is responsible for heavy queries.  

- **Calls**  
  - Number of times this query pattern has been executed.  
  - Use it to find high‑frequency queries that are good optimization candidates.  

- **Average time**  
  - Mean execution time across all runs of the query.  
  - Sort by this column to quickly identify consistently slow queries.  

- **Total time**  
  - Cumulative execution time for all runs.  
  - Useful for spotting queries that consume the most resources overall, even if each run is relatively fast.  

- **Query**  
  - The normalized SQL statement, with parameter values replaced by placeholders (for example, `$1`, `$2`).  
  - Grouping by normalized form lets you reason about one pattern instead of many similar parameterized queries. 

### 3.2 How It Works: pg_stat_statements

Give participants context on the underlying data source.

- The **Query performance** view is powered by the Postgres extension **`pg_stat_statements`**.  
- It runs on a system‑managed database and collects statistics for all queries against your instance, regardless of origin (SQL Editor, external clients, applications).  
- Data in `pg_stat_statements` is **not retained** when compute is suspended or restarted; collection resumes after restart.


> Note : Statistics collected by the pg_stat_statements extension are stored in memory and aren't retained when your Lakebase compute is suspended or restarted. For example, if your compute scales down due to inactivity, any existing statistics are lost. New statistics are gathered once your compute restarts.


##### Install Extension
`CREATE EXTENSION IF NOT EXISTS pg_stat_statements;`

##### Simple Select Statement
`SELECT * FROM pg_stat_statements LIMIT 10;`

#####Find Slowest queries
`SELECT
    query,
    calls,
    total_exec_time,
    mean_exec_time,
    (total_exec_time / calls) AS avg_time_ms
FROM pg_stat_statements
ORDER BY mean_exec_time DESC
LIMIT 20;`

## 4. Monitoring System Operations

This section explains how to use the **System operations** view to understand what the Lakebase platform is doing behind the scenes and to troubleshoot project or compute issues.

What Are System Operations?

- System operations are actions performed by the Lakebase platform on projects and resources.  
- They include both user‑initiated actions (for example, creating a branch or deleting a database) and system‑initiated actions (for example, suspending idle computes or checking availability).  
- Monitoring these operations helps you track project health, understand lifecycle events, and verify that actions completed successfully.  

Examples of operations you may see:

- Applying new configuration to a Lakebase object or resource (changing compute settings, creating/updating Postgres users and databases).  
- Updating configuration for storage‑level resources (such as changing the restore window).  
- Periodic **availability checks** that verify data in a branch and confirm that compute can start for that branch.  
- Creating a branch or setting up initial storage and the default branch when a project is created.  
- Deleting stored data when a Lakebase project is deleted.  
- Starting compute when activity requires resources (for example, connecting to a suspended compute).  
- Suspending compute after inactivity.  
- Attaching a project to storage, detaching it after extended idleness, and reattaching when a new request arrives.  
- Initiating branch archive and unarchive operations.  

### 4.1 How to access the System Operations view

To access the view: 

- In the Lakebase App, go to your project and open the **Monitoring** page.  
- Select the **System operations** tab.  

![system_operations.png](Includes/images/branching/lab5_system_operations.png)

What is included in the table:

- **Operation** – The specific action that was performed.  
- **Compute** – The compute instance where the operation occurred (if applicable).  
- **Operation status** – Current or final status of the operation.  
- **Duration** – How long the operation took.  
- **Date** – When the operation ran.

What are the status values and what do they mean:

- **OK** – Operation completed successfully.  
- **Scheduling** – Operation has been accepted and is waiting to run.  
- **In progress** – Operation is currently running.  
- **Error** – Operation failed.  

How to use this in troubleshooting:

- Filter or visually scan for **Error** status to find failed operations that may explain issues (for example, compute not starting or storage attach problems).  
- Look at **Duration** to spot unusually slow operations that might signal resource constraints or external dependencies.